In [1]:
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
import random
from typing import List, Tuple
from ultralytics import YOLO

In [2]:
def generate_yolo_seg_labels(
    boxes: List[Tuple[float, float, float, float]],
) -> List[str]:
    yolo_lines = []

    for x, y, w, h in boxes:
        cx = x
        cy = y
        points = [
            (cx - w / 2, cy - h / 2),
            (cx + w / 2, cy - h / 2),
            (cx + w / 2, cy + h / 2),
            (cx - w / 2, cy + h / 2),
        ]

        seg_points = [0.0]
        for px, py in points:
            seg_points.append(max(0.0, min(1.0, px)))
            seg_points.append(max(0.0, min(1.0, py)))

        yolo_line = " ".join([f"{p:.6f}" for p in seg_points])
        yolo_lines.append(yolo_line)

    return yolo_lines

In [3]:
class BloodCellTorchDataset(Dataset):
    def __init__(self, base_dataset_aug, base_dataset_real):
        self.base_dataset_aug = base_dataset_aug
        self.base_dataset_real = base_dataset_real
        self.data = {}

    def __len__(self):
        return len(self.base_dataset_aug) + len(self.base_dataset_real)

    def __getitem__(self, idx):
        if idx in self.data:
            return self.data[idx]
        
        if idx < len(self.base_dataset_real):
            gray_img, num_cells, boxes = self.base_dataset_real[idx]
        else:
            _, gray_img, _, num_cells, boxes = self.base_dataset_aug[idx - len(self.base_dataset_real)]

        self.data[idx] = gray_img, boxes, num_cells
        return self.data[idx]

In [4]:
from BloodCellDataset import slice_dataset, BloodCellDataset, PATCH_SIZE

np.random.seed(0)
random.seed(0)

bg_patches, cell_data, bg_masks, labeled_orig, orig_size = slice_dataset(
    train_dir="train",
    # sample_size=100,
    sample_size=2048,
    patch_size=PATCH_SIZE,
)

Slicing dataset: 100%|██████████| 1169/1169 [01:49<00:00, 10.65it/s]

Extracted 98843 background patches and 70383 cells


In [5]:
dataset = BloodCellDataset(
    orig_size=orig_size,
    img_size=(PATCH_SIZE * 4, PATCH_SIZE * 5),
    # n=100,
    n = 1280,
)
dataset.fit((bg_patches, bg_masks, cell_data, labeled_orig))

In [6]:
torch_dataset = BloodCellTorchDataset(dataset, labeled_orig)

In [7]:
train_set, val_set = torch.utils.data.random_split(
    torch_dataset, [0.8, 0.2]
)
len(train_set), len(val_set)

(1960, 489)

In [8]:
output_dir = "yolo_data_seg"
output_path = Path(output_dir)
img_dir = output_path / "images"
label_dir = output_path / "labels"

for split in ["train", "val"]:
    (img_dir / split).mkdir(parents=True, exist_ok=True)
    (label_dir / split).mkdir(parents=True, exist_ok=True)

split_name = "train"
for i, entry in enumerate(train_set):
    gray_img, boxes, num_cells = entry

    base_name = f"img_{i:05d}"
    gray_rgb = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2RGB)

    cv2.imwrite(str(img_dir / split_name / f"{base_name}.jpg"), gray_rgb)

    yolo_lines = generate_yolo_seg_labels(boxes)

    with open(label_dir / split_name / f"{base_name}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

split_name = "val"
for i, entry in enumerate(val_set):
    gray_img, boxes, num_cells = entry

    base_name = f"img_{i:05d}"
    gray_rgb = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2RGB)

    cv2.imwrite(str(img_dir / split_name / f"{base_name}.jpg"), gray_rgb)

    yolo_lines = generate_yolo_seg_labels(boxes)

    with open(label_dir / split_name / f"{base_name}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

In [9]:
yaml_content = {
    "path": str(output_path.absolute()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "blood_cell"},
}

yaml_path = output_path / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

In [10]:
model_size = "n" # 0.12
# model_size = "s" # 0.09
# model = YOLO(f"yolov8{model_size}-seg.pt")
model = YOLO(f"yolo26{model_size}-seg.pt")

In [11]:
epochs = 10
imgsz = 640
batch = 16
project = "runs"
name = "blood_cells_seg"

results = model.train(
    data=yaml_path,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    project=project,
    name=name,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

Ultralytics 8.4.51  Python-3.12.10 torch-2.9.0+rocmsdk20251116 CUDA:0 (AMD Radeon RX 9060 XT, 16304MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data_seg\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=blood_cells_seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

In [ ]:
err = 0.0

split_name = "val"
for i, entry in enumerate(val_set):
    gray_img, num_cells = entry[0], entry[2]
    
    base_name = f"img_{i:05d}"
    img_path = str(img_dir / split_name / f"{base_name}.jpg")
        
    results = model.predict(source=img_path)
    count = len(results[0].boxes)
    
    err += abs(count - num_cells) / num_cells

In [13]:
err / len(val_set)

0.08155532937477918